In [0]:
# 1. Parámetros dinámicos
dbutils.widgets.text("env", "dev", "Ambiente")
dbutils.widgets.text("carpeta", "transacciones", "Carpeta origen")

ambiente = dbutils.widgets.get("env")
carpeta_origen = dbutils.widgets.get("carpeta")

# 2. Asignación de tablas a iterar según la carpeta
if carpeta_origen == "catalogos":
    tablas = ["categorias", "clientes", "empleados", "productos", "proveedores", "subcategorias", "sucursales"]
elif carpeta_origen == "transacciones":
    tablas = ["ordenes_venta", "ordenes_venta_detalle", "facturas", "movimientos_inventario"]
else:
    tablas = []

storage_account = "adlsferreteria26"
container = "landing-zone"
ruta_origen = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{carpeta_origen}/"

# 3. Bucle de Ingesta para todas las tablas
for nombre_tabla in tablas:
    print(f"==== Iniciando ingesta Bronce: {nombre_tabla} ====")
    
    if carpeta_origen == "transacciones":
        filtro_archivos = f"{nombre_tabla}_[0-9]*.csv" 
    else:
        filtro_archivos = f"{nombre_tabla}.csv" 

    ruta_checkpoint = f"abfss://{container}@{storage_account}.dfs.core.windows.net/_checkpoints/{ambiente}/bronze/{nombre_tabla}"
    tabla_destino = f"ferreteria_{ambiente}.bronze.{nombre_tabla}"

    df_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", ruta_checkpoint)
        .option("pathGlobFilter", filtro_archivos)
        .option("header", "true")
        .load(ruta_origen)
    )

    (df_stream.writeStream
        .format("delta")
        .option("checkpointLocation", ruta_checkpoint)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(tabla_destino)
    )